<a href="https://colab.research.google.com/github/Nandhinisundarraj/project/blob/main/stock_price_spike_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**STOCK PRICE DETECTION:**

1.Problem Statement:

Predict whether a stock price will increase more than 2% the next day.

if yes -> 1(spike)

if no -> 0(no spike)

2.Data Collection:

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
stock = yf.download("RELIANCE.NS", start="2019-01-01", end="2024-12-31")

print(stock.head())
print(stock.tail())

/tmp/ipython-input-1711/2225326587.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock = yf.download("RELIANCE.NS", start="2019-01-01", end="2024-12-31")
[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open      Volume
Ticker     RELIANCE.NS RELIANCE.NS RELIANCE.NS RELIANCE.NS RELIANCE.NS
Date                                                                  
2019-01-01  498.490631  501.292179  493.643596  500.380573     9746670
2019-01-02  491.998230  501.158723  489.596917  495.600156    15628818
2019-01-03  485.928314  495.644647  484.749906  492.487392    16288287
2019-01-04  488.551910  491.131092  480.747723  487.996071    18516544
2019-01-07  491.264465  497.356638  489.596889  492.265005    12060290
Price             Close         High          Low         Open      Volume
Ticker      RELIANCE.NS  RELIANCE.NS  RELIANCE.NS  RELIANCE.NS RELIANCE.NS
Date                                                                      
2024-12-23  1217.437744  1222.318155  1208.373847  1210.166735    10052824
2024-12-24  1217.885864  1228.642950  1216.142826  1217.437703     6734917
2024-12-26  1211.710571  1222.816119  1209.419672  1219.3

3.Data Cleaning:

In [ ]:
stock.isnull().sum()

,,0
Price,Ticker,
Close,RELIANCE.NS,0
High,RELIANCE.NS,0
Low,RELIANCE.NS,0
Open,RELIANCE.NS,0
Volume,RELIANCE.NS,0


In [ ]:
stock.dropna()

Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,
2019-01-01,498.490631,501.292179,493.643596,500.380573,9746670
2019-01-02,491.998230,501.158723,489.596917,495.600156,15628818
2019-01-03,485.928314,495.644647,484.749906,492.487392,16288287
2019-01-04,488.551910,491.131092,480.747723,487.996071,18516544
2019-01-07,491.264465,497.356638,489.596889,492.265005,12060290
...,...,...,...,...,...
2024-12-23,1217.437744,1222.318155,1208.373847,1210.166735,10052824
2024-12-24,1217.885864,1228.642950,1216.142826,1217.437703,6734917


In [ ]:
stock = stock.reset_index()
stock.head()

Price,Date,Close,High,Low,Open,Volume
Ticker,,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
0,2019-01-01,498.490631,501.292179,493.643596,500.380573,9746670
1,2019-01-02,491.998230,501.158723,489.596917,495.600156,15628818
2,2019-01-03,485.928314,495.644647,484.749906,492.487392,16288287
3,2019-01-04,488.551910,491.131092,480.747723,487.996071,18516544
4,2019-01-07,491.264465,497.356638,489.596889,492.265005,12060290


In [ ]:
stock['Date'] = pd.to_datetime(stock['Date'])

In [ ]:
stock=stock.sort_values(by='Date')

5.Feature Engineering:

In [ ]:
stock['Daily_Return'] = stock['Close'].pct_change()

In [ ]:
stock['MA_10'] = stock['Close'].rolling(window=10).mean()
stock['MA_20'] = stock['Close'].rolling(window=20).mean()

In [ ]:
stock.columns = stock.columns.get_level_values(0)

In [ ]:
stock['BB_Upper'] = stock['MA_20'] + (2 * stock['Close'].rolling(20).std())
stock['BB_Lower'] = stock['MA_20'] - (2 * stock['Close'].rolling(20).std())

In [ ]:
delta = stock['Close'].diff()

gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

stock['RSI'] = 100 - (100 / (1 + rs))

In [ ]:
stock['Rolling_STD'] = stock['Close'].rolling(20).std()

In [ ]:
stock=stock.dropna()

In [ ]:
stock.head()

Price,Date,Close,High,Low,Open,Volume,Daily_Return,MA_10,MA_20,BB_Upper,BB_Lower,RSI,Next_Close,Next_Return,Spike,Rolling_STD
38,2019-02-25,547.983887,552.742046,542.803319,549.629229,17176522,-0.000041,549.647052,554.391824,582.008241,526.775407,35.755946,542.625427,-0.009778,0,13.808208
39,2019-02-26,542.625427,549.095600,536.288715,537.845064,22160532,-0.009778,548.179596,554.185043,582.102327,526.267760,33.579056,544.070618,0.002663,0,13.958642
40,2019-02-27,544.070618,553.586877,537.622708,546.093960,24308835,0.002663,546.716577,554.470752,581.817086,527.124418,28.489895,547.427979,0.006171,0,13.673167
41,2019-02-28,547.427979,551.341203,545.426899,548.628650,24688859,0.006171,547.021173,555.256726,580.684766,529.828686,34.327910,545.204712,-0.004061,0,12.714020
42,2019-03-01,545.204712,552.453091,543.514961,550.074011,17329606,-0.004061,546.202960,555.232269,580.699946,529.764593,35.783127,550.363037,0.009461,0,12.733838


In [ ]:
stock['Next_Close'] = stock['Close'].shift(-1)

stock['Next_Return'] = (stock['Next_Close'] - stock['Close']) / stock['Close']

/tmp/ipython-input-1711/3691587165.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stock['Next_Close'] = stock['Close'].shift(-1)
/tmp/ipython-input-1711/3691587165.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stock['Next_Return'] = (stock['Next_Close'] - stock['Close']) / stock['Close']


In [ ]:
stock['Spike'] = np.where(stock['Next_Return'] > 0.02, 1, 0)

/tmp/ipython-input-1711/807309201.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  stock['Spike'] = np.where(stock['Next_Return'] > 0.02, 1, 0)


In [ ]:
stock=stock.dropna()

In [ ]:
stock['Spike'].value_counts()

,count
Spike,
0,1301
1,139


6.Model Building:

In [ ]:
features = ['Daily_Return','MA_10','MA_20','Rolling_STD','RSI']
X = stock[features]

In [ ]:
y=stock['Spike']

In [ ]:
# 80% train, 20% test
split = int(len(stock) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [ ]:
# 80% train, 20% test
split = int(len(stock) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced')

In [ ]:
y_pred = model.predict(X_test)

7.Model Evaluation:

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9548611111111112

Confusion Matrix:
 [[275   0]
 [ 13   0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.98       275
           1       0.00      0.00      0.00        13

    accuracy                           0.95       288
   macro avg       0.48      0.50      0.49       288
weighted avg       0.91      0.95      0.93       288



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.95      1.00      0.98       275
           1       0.00      0.00      0.00        13

    accuracy                           0.95       288
   macro avg       0.48      0.50      0.49       288
weighted avg       0.91      0.95      0.93       288



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
